In [ ]:
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator
from scipy.special import expit

class KCNetTorch(BaseEstimator):
    """
    PyTorch version of KCNet with identical behavior.
    Structure:
        1. Initialization & Weight Setup
        2. Feature Transformation (Sparse Projection + Inhibition)
        3. Training & Prediction (Ridge Regression)
        4. Weight Updates (Optional Plasticity)
    """

    # ------------------------------
    # 1. Initialization & Weight Setup
    # ------------------------------
    def __init__(self, input_size, h_size=2000, num_glom_inputs=7,
                 decoder=None, alpha=1., W=None, S=None, gen_S=False):
        super().__init__()

        # Network dimensions
        self.input_size = input_size
        self.h_size = h_size
        self.num_glom_inputs = num_glom_inputs

        # Weights (converted to PyTorch tensors)
        self.W = torch.tensor(W, dtype=torch.float32) if W is not None else self.generate_sparse_weight(input_size)
        self.S = torch.tensor(S, dtype=torch.float32) if S is not None else None

        # Training components
        self.alpha = alpha
        self.decoder = decoder if decoder else RidgeClassifier(alpha=alpha, fit_intercept=False)

        if gen_S and self.S is None:
            self.S = self.weight_to_score()

    def generate_sparse_weight(self, num_feature):
        """Identical sparse binary weight matrix as before, but in PyTorch."""
        weight = torch.zeros((self.h_size, num_feature))
        for i in range(self.h_size):
            indices = torch.randperm(num_feature)[:self.num_glom_inputs]
            weight[i, indices] = 1.
        return weight

    # ------------------------------
    # 2. Feature Transformation
    # ------------------------------
    def generate_feature_map(self, x):
        """Same sparse projection + inhibition, now with PyTorch ops."""
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32)

        R_IN = torch.mm(x, self.W.T)  # Sparse projection
        R_OUT = torch.relu(R_IN - torch.mean(R_IN, dim=1, keepdim=True))  # Inhibition
        return R_OUT.numpy()  # For scikit-learn compatibility

    # ------------------------------
    # 3. Training & Prediction
    # ------------------------------
    def fit(self, x, y):
        """Identical training using RidgeClassifier."""
        if self.W is None:
            self.W = self.generate_sparse_weight(x.shape[1])
        if self.S is None:
            self.S = self.weight_to_score()
        self.decoder.fit(self.generate_feature_map(x), y)
        return self

    def predict(self, x):
        """Scikit-learn compatible prediction."""
        return self.decoder.predict(self.generate_feature_map(x))

    def predict_internal(self, x):
        """Sigmoid outputs for plasticity (converted to numpy for expit)."""
        self.rep_h = torch.tensor(self.generate_feature_map(x), dtype=torch.float32)
        beta = torch.tensor(self.decoder.coef_.T, dtype=torch.float32)
        return expit(torch.mm(self.rep_h, beta).numpy())

    # ------------------------------
    # 4. Weight Updates (Optional)
    # ------------------------------
    def update_W(self, x, y, lr):
        """Identical plasticity rule in PyTorch."""
        preds = torch.tensor(self.predict_internal(x), dtype=torch.float32)
        y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

        # Gradient calculation
        errors = torch.where(y > 0, preds - y, preds)
        beta = torch.tensor(self.decoder.coef_.T, dtype=torch.float32)
        d_hid = torch.mm(errors, beta.T)
        mask_h = (self.rep_h > 0).float()
        d_hid *= mask_h

        # Mask inputs
        mask_x = (torch.sum(self.W, dim=0) > 0).float()
        x_masked = torch.tensor(x, dtype=torch.float32) * mask_x

        # Update scores
        d_Wih = torch.mm(x_masked.T, d_hid).T
        self.S -= lr * d_Wih
        self.S = torch.clamp(self.S, min=-1, max=1)
        self.W = self.score_to_weight()

    # ------------------------------
    # Helper Methods (Unchanged Logic)
    # ------------------------------
    def weight_to_score(self, lb=-1, ub=0.02):
        """Same score initialization, now with PyTorch."""
        on_S = torch.rand(self.W.shape) * (ub - 0) + 0
        off_S = torch.rand(self.W.shape) * (0 - lb) + lb
        return torch.where(self.W > 0, on_S, off_S)

    def score_to_weight(self):
        """Binary thresholding."""
        return (self.S > 0).float()

    def append_score(self, n_add_hsize, lb=-1, ub=0.02):
        """Identical hidden layer expansion."""
        new_S = torch.rand((n_add_hsize, self.input_size)) * (ub - lb) + lb
        self.S = torch.cat([self.S, new_S], dim=0)
        self.h_size += n_add_hsize
        self.W = self.score_to_weight()

    # Scikit-learn compatibility
    def get_params(self, deep=True):
        return {'input_size': self.input_size, 'h_size': self.h_size, 'alpha': self.alpha}

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.linear_model import RidgeClassifier

# Parameters
input_dim = 50      # Input features
hidden_dim = 2000   # Kenyon cells
output_dim = 10     # Classes
connections_per_hidden = 7  # Sparsity
lambda_reg = 0.1    # Ridge regularization

n_runs = 10
results = []

for run in range(n_runs):
    # Initialize KCNetTorch (PyTorch version)
    kcnet = KCNetTorch(input_size=input_dim, h_size=hidden_dim,
                      num_glom_inputs=connections_per_hidden, alpha=lambda_reg)

    # Example data (identical to original test)
    X = torch.rand(100, input_dim)  # 100 samples
    y = torch.randint(0, output_dim, (100,))  # Random labels

    # Convert y to one-hot for RidgeClassifier (not needed for RidgeClassifier in scikit-learn)
    # y_one_hot = torch.nn.functional.one_hot(y, num_classes=output_dim).float()

    # Step 1: Train the model (via scikit-learn's RidgeClassifier)
    # Pass the raw input data X.numpy() to fit
    kcnet.fit(X.numpy(), y.numpy())

    # Step 2: Predictions
    # Pass the raw input data X.numpy() to predict as well
    predictions = kcnet.predict(X.numpy())

    # Step 3: Evaluation
    accuracy = accuracy_score(y.numpy(), predictions)
    print(f"Run {run + 1} Accuracy: {accuracy * 100:.2f}%")
    results.append(accuracy)

# Calculate statistics
average_result = np.mean(results)
std_deviation = np.std(results)

print(f"\nFinal Results (PyTorch Version):")
print(f"Average Accuracy: {average_result:.4f} ± {std_deviation:.4f}")

Run 1 Accuracy: 100.00%
Run 2 Accuracy: 100.00%
Run 3 Accuracy: 100.00%
Run 4 Accuracy: 100.00%
Run 5 Accuracy: 100.00%
Run 6 Accuracy: 100.00%
Run 7 Accuracy: 100.00%
Run 8 Accuracy: 100.00%
Run 9 Accuracy: 100.00%
Run 10 Accuracy: 100.00%

Final Results (PyTorch Version):
Average Accuracy: 1.0000 ± 0.0000


more cleaner

In [ ]:
import torch
import torch.nn as nn
from sklearn.linear_model import RidgeClassifier
import numpy as np

class KCNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=2000, connections_per_hidden=7, alpha=1.0):
        """
        Simplified PyTorch version of your second KCNet code.
        Follows the same structure as your first implementation.
        """
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.connections_per_hidden = connections_per_hidden
        self.alpha = alpha

        # Sparse projection matrix (binary)
        self.projection = self._generate_sparse_projection()

        # Score matrix (for plasticity)
        self.scores = self._initialize_scores()

        # Output layer (ridge regression)
        self.decoder = RidgeClassifier(alpha=alpha, fit_intercept=False)

    def _generate_sparse_projection(self):
        """Create binary sparse weight matrix (fixed connections)."""
        weight = torch.zeros((self.hidden_dim, self.input_dim))
        for i in range(self.hidden_dim):
            indices = torch.randperm(self.input_dim)[:self.connections_per_hidden]
            weight[i, indices] = 1
        return weight

    def _initialize_scores(self, lb=-1, ub=0.02):
        """Initialize score matrix for plasticity."""
        on_scores = torch.rand((self.hidden_dim, self.input_dim)) * (ub - 0) + 0
        off_scores = torch.rand((self.hidden_dim, self.input_dim)) * (0 - lb) + lb
        return torch.where(self.projection > 0, on_scores, off_scores)

    def forward(self, x):
        """Forward pass: sparse projection + inhibition."""
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x).float()

        # Sparse projection
        hidden_pre = torch.mm(x, self.projection.T)

        # Global inhibition
        hidden_post = torch.relu(hidden_pre - torch.mean(hidden_pre, dim=1, keepdim=True))

        return hidden_post.numpy()  # For scikit-learn compatibility

    def train_output(self, X, y):
        """Train output layer with ridge regression."""
        features = self.forward(X)
        self.decoder.fit(features, y)
        return self

    def predict(self, X):
        """Predict using trained decoder."""
        features = self.forward(X)
        return self.decoder.predict(features)

    def update_weights(self, X, y, lr=0.01):
        """Optional plasticity update (matches original update_W)."""
        # Convert to torch
        X_t = torch.from_numpy(X).float()
        y_t = torch.from_numpy(y).float().view(-1, 1)

        # Forward pass
        hidden_act = torch.mm(X_t, self.projection.T)
        hidden_out = torch.relu(hidden_act - torch.mean(hidden_act, dim=1, keepdim=True))

        # Get predictions
        beta = torch.from_numpy(self.decoder.coef_.T).float()
        preds = torch.sigmoid(torch.mm(hidden_out, beta))

        # Compute gradient
        errors = torch.where(y_t > 0, preds - y_t, preds)
        d_hid = torch.mm(errors, beta.T)
        mask_h = (hidden_out > 0).float()
        d_hid *= mask_h

        # Update scores
        mask_x = (torch.sum(self.projection, dim=0) > 0).float()
        x_masked = X_t * mask_x
        d_Wih = torch.mm(x_masked.T, d_hid).T

        self.scores -= lr * d_Wih
        self.scores = torch.clamp(self.scores, min=-1, max=1)
        self.projection = (self.scores > 0).float()

In [ ]:
# Parameters
input_dim = 50
hidden_dim = 2000
output_dim = 10
connections = 7
alpha = 0.1
n_runs = 10

results = []
for run in range(n_runs):
    # Initialize model
    model = KCNet(input_dim, hidden_dim, connections, alpha)

    # Generate data
    X = np.random.rand(100, input_dim)
    y = np.random.randint(0, output_dim, 100)

    # Train and predict
    model.train_output(X, y)
    preds = model.predict(X)

    # Evaluate
    acc = accuracy_score(y, preds)
    print(f"Run {run+1}: {acc*100:.2f}%")
    results.append(acc)

print(f"\nFinal Accuracy: {np.mean(results):.4f} ± {np.std(results):.4f}")

Run 1: 100.00%
Run 2: 100.00%
Run 3: 100.00%
Run 4: 100.00%
Run 5: 100.00%
Run 6: 100.00%
Run 7: 100.00%
Run 8: 100.00%
Run 9: 100.00%
Run 10: 100.00%

Final Accuracy: 1.0000 ± 0.0000


A bit biological version,inhibition/activation